# Phase 1 completion on Kaggle: seed 44 (HI -> DE -> EN)

This notebook completes the unfinished third seed of the existing Phase 1 follow-up. It uses the original seed-44 training selection, fixed test selection, Hindi -> German -> English order, model, prompt, and LoRA settings. Seeds 42 and 43 are loaded from the prior saved-results ZIP and are **not retrained**.

## Kaggle setup

1. Create a **private Kaggle Dataset** containing these two files, then attach it to this notebook as an input. Kaggle may show them as extracted folders instead of `.zip` files; the notebook accepts either form and automatically uses one copy if duplicate extracted inputs are identical:
   - `phase1_xnli_multiseed_colab_package.zip`
   - `continual_learning_phase1_hi_de_en-20261110T104926Z-1-001.zip` (the organized Phase 1 results ZIP)
2. In Notebook options, enable a GPU (T4/P100 or better) and turn Internet on. GPU availability is subject to your Kaggle account's quota and current availability; do not use account rotation to bypass limits.
3. Accept access to `meta-llama/Llama-3.2-1B` and add a Kaggle secret named `HF_TOKEN`.
4. Run all cells. Results and checkpoints are written under `/kaggle/working/phase1_completion` (Kaggle inputs are read-only).
5. After a successful run, use **Save Version** with notebook output enabled, and download `phase1_completion_outputs.zip` for a separate backup.

If you uploaded the ZIPs twice, remove duplicate attached datasets or keep just one copy. The original seed-44 ledger is incomplete and is never overwritten. The notebook starts a fresh attempt and updates a copy of the group summary only after all three seeds pass result validation. If a Kaggle runtime disconnects before completion, rerunning starts a fresh attempt; it does not claim to resume a partially trained stage.


In [ ]:
%pip -q install "datasets>=3.0" "transformers>=4.46" "peft>=0.13" accelerate bitsandbytes wandb


In [ ]:
import os
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Enable a GPU in Kaggle Notebook options, then rerun this cell.")

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a Kaggle secret named HF_TOKEN before running this notebook.")
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
del hf_token

ENABLE_WANDB = False
if ENABLE_WANDB:
    wandb_key = secrets.get_secret("WANDB_API_KEY")
    if not wandb_key:
        raise RuntimeError("Add a Kaggle secret named WANDB_API_KEY before enabling W&B.")
    os.environ["WANDB_API_KEY"] = wandb_key
    del wandb_key

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print("Hugging Face authentication completed; credentials were not printed.")


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import sys
import hashlib

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/phase1_completion")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
GROUP_ID = "xnli-hi-de-en-multiseed-train1000-test500-20261110T045241Z"


def find_input_archive_or_folder(filename, folder_check, identity_files):
    archives = sorted(path for path in INPUT_ROOT.rglob(filename) if path.is_file())
    if len(archives) > 1:
        raise FileNotFoundError(f"Found duplicate input archives named {filename}: {archives}")
    if archives:
        return archives[0]

    folders = sorted(path for path in INPUT_ROOT.rglob("*") if path.is_dir() and folder_check(path))
    if not folders:
        raise FileNotFoundError(
            f"Expected {filename} or its extracted folder under {INPUT_ROOT}; "
            "expand the Input panel and check the attached dataset contents."
        )
    if len(folders) > 1:
        signatures = []
        for folder in folders:
            files = sorted(identity_files(folder))
            if not files or any(not file.is_file() for file in files):
                raise FileNotFoundError(f"Could not verify duplicate input folder: {folder}")
            signatures.append(tuple(
                (str(file.relative_to(folder)), file.stat().st_size, hashlib.sha256(file.read_bytes()).hexdigest())
                for file in files
            ))
        if any(signature != signatures[0] for signature in signatures[1:]):
            raise RuntimeError(f"Found multiple non-identical extracted inputs for {filename}: {folders}")
        print(f"Found {len(folders)} identical extracted copies of {filename}; using {folders[0]}")
    return folders[0]


SOURCE_PACKAGE_PATH = find_input_archive_or_folder(
    "phase1_xnli_multiseed_colab_package.zip",
    lambda path: (path / "shared" / "continual_learning_lora" / "experiment.py").is_file(),
    lambda path: [file for file in (path / "shared" / "continual_learning_lora").rglob("*") if file.is_file()] if (path / "shared" / "continual_learning_lora").is_dir() else (),
)
RESULTS_PACKAGE_PATH = find_input_archive_or_folder(
    "continual_learning_phase1_hi_de_en-20261110T104926Z-1-001.zip",
    lambda path: (
        path.name == GROUP_ID
        and (path / "multiseed_summary.json").is_file()
        and all((path / f"seed-{seed}" / "evaluation" / "evaluation_results.json").is_file() for seed in (42, 43, 44))
    ),
    lambda path: [
        path / "multiseed_summary.json",
        *(path / f"seed-{seed}" / "evaluation" / "evaluation_results.json" for seed in (42, 43, 44)),
    ],
)

PACKAGE_DIR = OUTPUT_ROOT / "source_package"
if SOURCE_PACKAGE_PATH.is_file():
    PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
    with ZipFile(SOURCE_PACKAGE_PATH) as archive:
        root = PACKAGE_DIR.resolve()
        for member in archive.infolist():
            destination = (PACKAGE_DIR / member.filename).resolve()
            if destination != root and root not in destination.parents:
                raise ValueError("The source ZIP contains an unsafe path.")
        archive.extractall(PACKAGE_DIR)
    PACKAGE_SOURCE_ROOT = PACKAGE_DIR
else:
    PACKAGE_SOURCE_ROOT = SOURCE_PACKAGE_PATH

if not (PACKAGE_SOURCE_ROOT / "shared" / "continual_learning_lora" / "experiment.py").is_file():
    raise FileNotFoundError(f"The Phase 1 experiment package was not found in {PACKAGE_SOURCE_ROOT}.")
sys.path.insert(0, str(PACKAGE_SOURCE_ROOT))

if RESULTS_PACKAGE_PATH.is_dir():
    RESULTS_GROUP_ROOT = RESULTS_PACKAGE_PATH
else:
    RESULTS_GROUP_ROOT = None

print("Source package input:", SOURCE_PACKAGE_PATH)
print("Saved results input:", RESULTS_PACKAGE_PATH)
print("Input format:", "ZIP archive" if SOURCE_PACKAGE_PATH.is_file() else "Kaggle-extracted folders")
print("Phase 1 experiment package loaded.")


In [ ]:
import json
import math
from pathlib import Path
from zipfile import ZipFile

EXPERIMENT_ID = f"{GROUP_ID}-seed44"
LANGUAGES = ["hi", "de", "en"]
SEED = 44
TRAIN_SAMPLES_PER_LANGUAGE = 1000
TEST_SAMPLES_PER_LANGUAGE = 500
TEST_SPLIT_SEED = 42
DATASET_ID = "facebook/xnli"
MODEL_ID = "meta-llama/Llama-3.2-1B"

GROUP_OUTPUT = OUTPUT_ROOT / GROUP_ID
PRIOR_RESULTS_DIR = GROUP_OUTPUT / "prior_results"
PRIOR_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ATTEMPTS_ROOT = GROUP_OUTPUT / "seed-44-attempts"
ATTEMPTS_ROOT.mkdir(parents=True, exist_ok=True)

ARCHIVE_PREFIX = f"continual_learning_phase1_hi_de_en/{GROUP_ID}/"


def copy_saved_result(suffix, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if RESULTS_PACKAGE_PATH.is_file():
        with ZipFile(RESULTS_PACKAGE_PATH) as archive:
            member = ARCHIVE_PREFIX + suffix
            if member not in archive.namelist():
                raise FileNotFoundError(f"Expected archived member {member}.")
            destination.write_bytes(archive.read(member))
    else:
        source = RESULTS_GROUP_ROOT / suffix
        if not source.is_file():
            raise FileNotFoundError(f"Expected saved result file {source}.")
        destination.write_bytes(source.read_bytes())
    return destination


previous_summary_path = copy_saved_result(
    "multiseed_summary.json", PRIOR_RESULTS_DIR / "multiseed_summary.original.json"
)
seed_paths = {
    seed: copy_saved_result(
        f"seed-{seed}/evaluation/evaluation_results.json",
        PRIOR_RESULTS_DIR / f"seed-{seed}-evaluation_results.json",
    )
    for seed in (42, 43)
}
incomplete_seed44_path = copy_saved_result(
    "seed-44/evaluation/evaluation_results.json",
    PRIOR_RESULTS_DIR / "seed-44-incomplete-evaluation_results.json",
)

with incomplete_seed44_path.open(encoding="utf-8") as file:
    incomplete_seed44 = json.load(file)
if incomplete_seed44.get("experiment_id") != EXPERIMENT_ID:
    raise ValueError("The archived seed-44 ledger belongs to a different experiment.")
if incomplete_seed44.get("task_order") != LANGUAGES:
    raise ValueError("The archived seed-44 ledger has a different task order.")
if incomplete_seed44.get("run_metadata", {}).get("seed") != SEED:
    raise ValueError("The archived seed-44 ledger has a different seed.")
if any(value is not None for row in incomplete_seed44.get("performance_matrix", []) for value in row):
    raise RuntimeError("The archived seed-44 ledger contains stage results; inspect it before starting another run.")


def load_complete_run(path, seed):
    with Path(path).open(encoding="utf-8") as file:
        record = json.load(file)
    if record.get("experiment_id") != f"{GROUP_ID}-seed{seed}":
        raise ValueError(f"Wrong experiment ID in {path}.")
    if record.get("task_order") != LANGUAGES:
        raise ValueError(f"Wrong task order in {path}.")
    if record.get("run_metadata", {}).get("seed") != seed:
        raise ValueError(f"Wrong seed in {path}.")
    matrix = record.get("performance_matrix")
    if not isinstance(matrix, list) or len(matrix) != len(LANGUAGES):
        raise ValueError(f"Incomplete stage matrix in {path}.")
    if any(not isinstance(row, list) or len(row) != len(LANGUAGES) for row in matrix):
        raise ValueError(f"Malformed stage matrix in {path}.")
    if any(not isinstance(value, (int, float)) or not math.isfinite(value) or not 0 <= value <= 1 for row in matrix for value in row):
        raise ValueError(f"The matrix contains missing or invalid accuracy values in {path}.")
    metrics = record.get("metrics")
    if not isinstance(metrics, dict) or any(not isinstance(metrics.get(name), (int, float)) or not math.isfinite(metrics[name]) for name in ("ACC", "BWT", "FWT", "F")):
        raise ValueError(f"The metrics are incomplete in {path}.")
    baselines = record.get("baseline_accuracies")
    if not isinstance(baselines, list) or len(baselines) != len(LANGUAGES) or any(not isinstance(value, (int, float)) or not math.isfinite(value) for value in baselines):
        raise ValueError(f"The zero-shot baselines are incomplete in {path}.")
    return {
        "seed": seed,
        "experiment_id": record["experiment_id"],
        "matrix": matrix,
        "metrics": metrics,
        "evaluation_json": str(path),
    }


seed_results = [load_complete_run(seed_paths[seed], seed) for seed in (42, 43)]
print("Validated saved seeds:", [item["seed"] for item in seed_results])
print("Seed 44 had only zero-shot baselines; it will be trained in a new attempt folder.")


In [ ]:
from datasets import load_dataset

LABEL_NAMES = {0: "entailment", 1: "neutral", 2: "contradiction"}

def format_xnli_row(example):
    label_id = int(example["label"])
    if label_id not in LABEL_NAMES:
        raise ValueError(f"Unexpected XNLI label id: {label_id}")
    prompt = (
        "Task: Natural language inference\n"
        f"Premise: {example['premise']}\n"
        f"Hypothesis: {example['hypothesis']}\n"
        "Choose one label: entailment, neutral, contradiction\n"
        "Answer:"
    )
    return {"prompt": prompt, "target": " " + LABEL_NAMES[label_id]}

train_data = {}
test_data = {}
for language in LANGUAGES:
    train_raw = load_dataset(DATASET_ID, language, split="train")
    test_raw = load_dataset(DATASET_ID, language, split="test")
    train_count = min(TRAIN_SAMPLES_PER_LANGUAGE, len(train_raw))
    test_count = min(TEST_SAMPLES_PER_LANGUAGE, len(test_raw))
    if train_count != TRAIN_SAMPLES_PER_LANGUAGE or test_count != TEST_SAMPLES_PER_LANGUAGE:
        raise ValueError(f"Unexpected XNLI split size for {language}: train={train_count}, test={test_count}.")
    train_subset = train_raw.shuffle(seed=SEED).select(range(train_count))
    test_data[language] = test_raw.shuffle(seed=TEST_SPLIT_SEED).select(range(test_count))
    train_data[language] = train_subset.map(
        format_xnli_row,
        remove_columns=train_subset.column_names,
        desc=f"Formatting {language} training examples for seed {SEED}",
    )
    if set(train_data[language].column_names) != {"prompt", "target"}:
        raise ValueError(f"Unexpected formatted training columns for {language}.")
    print(f"{language}: train={len(train_data[language])}, fixed test={len(test_data[language])}")
print("Data prepared using the original seed-44 selection and fixed test-selection rules.")


In [ ]:
import gc
from datetime import datetime, timezone
from transformers import set_seed
from shared.continual_learning_lora import LlamaLoRATrainer, run_sequential_experiment

completed_attempts = []
for result_path in ATTEMPTS_ROOT.glob("*/evaluation/evaluation_results.json"):
    try:
        completed_attempts.append(load_complete_run(result_path, SEED))
    except (ValueError, KeyError, TypeError) as exc:
        print(f"Ignoring incomplete attempt {result_path}: {exc}")

if completed_attempts:
    seed44_result = sorted(completed_attempts, key=lambda item: item["evaluation_json"])[-1]
    print("Using a previously completed attempt:", seed44_result["evaluation_json"])
else:
    attempt_id = datetime.now(timezone.utc).strftime("attempt-%Y%m%dT%H%M%SZ")
    attempt_root = ATTEMPTS_ROOT / attempt_id
    if attempt_root.exists():
        raise FileExistsError(f"Attempt folder already exists: {attempt_root}. Rerun to create a fresh attempt.")
    checkpoint_dir = attempt_root / "checkpoints"
    evaluation_dir = attempt_root / "evaluation"
    attempt_root.mkdir(parents=True, exist_ok=False)

    set_seed(SEED)
    trainer = LlamaLoRATrainer(
        model_name=MODEL_ID,
        output_dir=checkpoint_dir,
        task_mode="causal_lm",
        max_length=512,
        learning_rate=2e-4,
        train_batch_size=1,
        eval_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        seed=SEED,
        lora_rank=8,
        lora_alpha=16,
        lora_dropout=0.05,
        load_in_4bit=True,
        gradient_checkpointing=True,
        token=True,
    )
    trainer.load_model_and_tokenizer()
    trainer.model.print_trainable_parameters()
    result = run_sequential_experiment(
        trainer,
        task_order=LANGUAGES,
        train_datasets=train_data,
        test_datasets=test_data,
        experiment_id=EXPERIMENT_ID,
        output_dir=evaluation_dir,
        dataset_name=DATASET_ID,
        task_name="xnli",
        run_metadata={
            "phase": 1,
            "method": "sequential_finetuning_no_replay",
            "seed": SEED,
            "seed_group": GROUP_ID,
            "test_split_seed": TEST_SPLIT_SEED,
            "model": MODEL_ID,
            "train_samples_per_language": TRAIN_SAMPLES_PER_LANGUAGE,
            "test_samples_per_language": TEST_SAMPLES_PER_LANGUAGE,
            "test_split": "test",
            "prompt_protocol": "phase1_b0_nli_prompt_candidate_label_log_likelihood",
            "attempt_id": attempt_id,
            "recovered_from_incomplete_seed44": True,
        },
        wandb_project=("asymmetric-interference-continual-learning" if ENABLE_WANDB else None),
        wandb_mode="online",
    )
    seed44_result = load_complete_run(result["ledger"].json_path, SEED)
    del result, trainer
    gc.collect()
    torch.cuda.empty_cache()
    print("Seed 44 completed:", seed44_result["evaluation_json"])

seed_results.append(seed44_result)
seed_results.sort(key=lambda item: item["seed"])
if [item["seed"] for item in seed_results] != [42, 43, 44]:
    raise ValueError("Expected exactly one validated result for seeds 42, 43, and 44.")


In [ ]:
import json
import os
import shutil
import statistics
from datetime import datetime, timezone

matrices = [item["matrix"] for item in seed_results]
mean_matrix = [
    [statistics.mean(matrix[row][column] for matrix in matrices) for column in range(len(LANGUAGES))]
    for row in range(len(LANGUAGES))
]
std_matrix = [
    [statistics.stdev(seed_matrix[row][column] for seed_matrix in matrices) for column in range(len(LANGUAGES))]
    for row in range(len(LANGUAGES))
]
seed_pattern_results = []
for item in seed_results:
    matrix = item["matrix"]
    final_row = matrix[-1]
    diagonal = [matrix[index][index] for index in range(len(LANGUAGES))]
    previous_final_mean = statistics.mean(final_row[:-1])
    previous_learned_mean = statistics.mean(diagonal[:-1])
    last_language_strong = final_row[-1] >= previous_final_mean
    earlier_languages_degraded = previous_final_mean < previous_learned_mean
    seed_pattern_results.append({
        "seed": item["seed"],
        "last_seen_language_strong_relative_to_previous_mean": last_language_strong,
        "earlier_languages_degraded_on_average": earlier_languages_degraded,
        "checkpoint_pattern_passed": last_language_strong and earlier_languages_degraded,
    })

final_mean = mean_matrix[-1]
first_seen_mean = [statistics.mean(matrix[index][index] for matrix in matrices) for index in range(len(LANGUAGES))]
report = {
    "experiment_group_id": GROUP_ID,
    "task_order": LANGUAGES,
    "seeds": [42, 43, 44],
    "train_examples_per_language": TRAIN_SAMPLES_PER_LANGUAGE,
    "test_examples_per_language": TEST_SAMPLES_PER_LANGUAGE,
    "fixed_test_split_seed": TEST_SPLIT_SEED,
    "mean_performance_matrix_rows_after_each_stage": mean_matrix,
    "std_performance_matrix_rows_after_each_stage": std_matrix,
    "mean_final_accuracies_by_language": dict(zip(LANGUAGES, final_mean)),
    "mean_first_seen_accuracies_by_language": dict(zip(LANGUAGES, first_seen_mean)),
    "per_seed_results": seed_results,
    "per_seed_checkpoint_checks": seed_pattern_results,
    "checkpoint_pattern_pass_rate": sum(item["checkpoint_pattern_passed"] for item in seed_pattern_results) / len(seed_pattern_results),
    "checkpoint_pattern_passed_all_seeds": all(item["checkpoint_pattern_passed"] for item in seed_pattern_results),
    "interpretation_note": "Three completed seeds provide a preliminary repeatability check, not a definitive scientific conclusion.",
}

summary_path = GROUP_OUTPUT / "multiseed_summary.json"
backup_path = GROUP_OUTPUT / f"multiseed_summary.before-kaggle-completion-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')}.json"
if backup_path.exists():
    raise FileExistsError(f"Refusing to overwrite summary backup: {backup_path}")
shutil.copy2(previous_summary_path, backup_path)
temporary_path = summary_path.with_suffix(".json.tmp")
temporary_path.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")
os.replace(temporary_path, summary_path)

print("Updated summary:", summary_path)
print("Previous summary preserved:", backup_path)
print("Per-seed checkpoint checks:", json.dumps(seed_pattern_results, indent=2))
print("Checkpoint pass rate:", report["checkpoint_pattern_pass_rate"])
if report["checkpoint_pattern_passed_all_seeds"]:
    print("All seeds met the pilot criterion. This is preliminary evidence, not definitive reproduction.")
else:
    print("Three-seed run is complete, but the checkpoint pattern was not met by every seed. Do not claim robust reproduction.")


## Preserve the run

The run's adapters and evaluation ledger are in `/kaggle/working/phase1_completion/<group>/seed-44-attempts/`. The validated summary and a copy of the previous summary are beside that folder. Save a notebook version with output enabled, and download the archive created below. Keep the two input ZIPs unchanged for provenance.


In [ ]:
import shutil
from IPython.display import FileLink, display

archive_base = Path("/kaggle/working/phase1_completion_outputs")
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_ROOT))
print("Output archive:", archive_path)
print("Size (MiB):", round(archive_path.stat().st_size / (1024 * 1024), 1))
display(FileLink(str(archive_path)))
